# SOFC Voltage Forecasting — Colab Runner (LSTM, Seq2Seq LSTM, TCN)

Notebook này train **LSTM, Seq2Seq LSTM, TCN** cho target **`V`** (điện áp SOFC) trên Colab với GPU miễn phí. Random Forest và XGBoost chạy riêng ở local (không cần GPU, xem `src/main.py`).

**Cách hoạt động:** `E:\sofc` giờ đã là git repo (`github.com/trungthanh-dev/SOFC`, public), nên notebook này **clone/pull thẳng vào Google Drive** — giống hệt cách `E:\FCF\FCF_Colab.ipynb` hoạt động. File data thô `data/raw/DataTime_export.csv` (16MB) đã được commit thẳng vào repo (khác FCF — bên đó `data_clean_power/*.parquet` bị `.gitignore` nên phải upload tay), nên **không cần upload bất kỳ file nào thủ công** — chỉ cần mount Drive rồi chạy lần lượt các cell.

**Trước khi chạy:**
1. `Runtime -> Change runtime type -> GPU (T4)`.
2. Repo là public nên để `GITHUB_TOKEN = ""` ở cell cấu hình bên dưới là chạy được ngay.


## 1. Mount Google Drive

In [48]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Cấu hình repo

Repo public nên để `GITHUB_TOKEN = ""` là đủ. Chỉ cần điền token nếu sau này bạn chuyển repo sang Private (GitHub -> Settings -> Developer settings -> Personal access tokens -> Generate new token, tick quyền `repo`).


In [49]:
GITHUB_USERNAME = "trungthanh-dev"
GITHUB_REPO = "SOFC"
GITHUB_TOKEN = ""  # dán token vào đây nếu repo Private, để trống nếu Public

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/SOFC"
SRC_DIR = f"{DRIVE_PROJECT_DIR}/src"


## 3. Clone (lần đầu) hoặc Pull (các lần sau)

Cell này tự phát hiện: nếu `DRIVE_PROJECT_DIR` chưa tồn tại trên Drive thì clone, nếu đã tồn tại thì chỉ `git pull` để lấy code mới nhất từ GitHub — bao gồm cả `data/raw/DataTime_export.csv` vì file này đã nằm trong git.


In [50]:
import os

if GITHUB_TOKEN:
    remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git"
else:
    remote_url = f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git"

if not os.path.exists(DRIVE_PROJECT_DIR):
    print("Chưa có project trên Drive -> clone lần đầu...")
    !git clone {remote_url} "{DRIVE_PROJECT_DIR}"
else:
    print("Project đã có trên Drive -> pull code mới nhất...")
    %cd {DRIVE_PROJECT_DIR}
    !git pull

%cd {DRIVE_PROJECT_DIR}
!git log --oneline -5


Project đã có trên Drive -> pull code mới nhất...
/content/drive/MyDrive/SOFC
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 12 (delta 5), reused 12 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 9.30 KiB | 43.00 KiB/s, done.
From https://github.com/trungthanh-dev/SOFC
   1b703f7..e0942c1  main       -> origin/main
Updating 1b703f7..e0942c1
Fast-forward
 notebooks/SOFC_Colab_Forecasting.ipynb     |  24 +++++
 notes/SOFC_data_notes.md                   |  82 +++++++++++++++-
 src/main_delta_vs_persistence_multiseed.py | 153 +++++++++++++++++++++++++++++
 3 files changed, 257 insertions(+), 2 deletions(-)
 create mode 100644 src/main_delta_vs_persistence_multiseed.py
/content/drive/MyDrive/SOFC
e0942c1 (HEAD -> main, origin/main, origin/HEAD) Add multi-seed validation for the flagship claim (delta-target V vs persistence)
c4a6c91 Resolve literature novelty question defin

## 4. Kiểm tra file cần thiết đã có chưa

Vì clone/pull thẳng từ GitHub, bước này chỉ để xác nhận không có gì bị thiếu (ví dụ do `.gitignore` lỡ loại nhầm) trước khi chạy pipeline — không cần upload thủ công.


In [51]:
required = [
    f"{DRIVE_PROJECT_DIR}/data/raw/DataTime_export.csv",
    f"{SRC_DIR}/config.py",
    f"{SRC_DIR}/preprocessing.py",
    f"{SRC_DIR}/features.py",
    f"{SRC_DIR}/windowing.py",
    f"{SRC_DIR}/diagnostics.py",
    f"{SRC_DIR}/main_lstm.py",
    f"{SRC_DIR}/main_tcn.py",
    f"{SRC_DIR}/main_seq2seq.py",
    f"{SRC_DIR}/main_lstm_delta.py",
    f"{SRC_DIR}/main_tcn_delta.py",
    f"{SRC_DIR}/main_seq2seq_delta.py",
    f"{SRC_DIR}/main_lstm_power.py",
    f"{SRC_DIR}/main_tcn_power.py",
    f"{SRC_DIR}/main_seq2seq_power.py",
    f"{SRC_DIR}/main_lstm_power_delta.py",
    f"{SRC_DIR}/main_tcn_power_delta.py",
    f"{SRC_DIR}/main_seq2seq_power_delta.py",
    f"{SRC_DIR}/main_lstm_delta_given_i.py",
    f"{SRC_DIR}/main_tcn_delta_given_i.py",
    f"{SRC_DIR}/main_seq2seq_delta_given_i.py",
    f"{SRC_DIR}/main_given_i_multiseed.py",
    f"{SRC_DIR}/main_delta_vs_persistence_multiseed.py",
    f"{SRC_DIR}/models/__init__.py",
    f"{SRC_DIR}/models/lstm.py",
    f"{SRC_DIR}/models/tcn.py",
    f"{SRC_DIR}/models/seq2seq_lstm.py",
]
missing = [p for p in required if not os.path.exists(p)]

if missing:
    print("THIẾU các file sau trên Drive (kiểm tra lại git clone/pull ở mục 3):")
    for p in missing:
        print(" -", p)
else:
    print("Đã có đủ file cần thiết. Sẵn sàng chạy tiếp.")


Đã có đủ file cần thiết. Sẵn sàng chạy tiếp.


## 5. Cài đặt thư viện

`torch`, `pandas`, `scikit-learn` đã có sẵn trên Colab. `xgboost` không cần cho notebook này (RF/XGBoost chạy local) nhưng `models/xgboost_model.py` bị import gián tiếp khi Python quét `models/` package -- cài luôn cho chắc, không tốn thời gian đáng kể.


In [52]:
!pip install -q xgboost


## 6. Kiểm tra GPU

In [53]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Không có GPU -> Runtime -> Change runtime type -> GPU (T4), rồi Runtime -> Restart session và chạy lại từ đầu.")


CUDA available: True
GPU: Tesla T4


## 7. Chạy pipeline (khuyến nghị: 1 cell chạy hết — `Runtime -> Run all` là xong)

Cell dưới đây chạy tuần tự cả 12 script (V + Power, raw + Delta-Target, LSTM/TCN/Seq2Seq) trong 1 lần bấm — không cần bấm từng cell theo đúng thứ tự nữa. Mỗi script vẫn tự lưu model/predictions/report vào `outputs/` trên Drive như trước; nếu 1 script lỗi, cell in rõ tên script lỗi rồi **tiếp tục chạy các script còn lại** (không dừng hết cả pipeline vì 1 lỗi).

Muốn train lại **đúng 1 model cụ thể** (VD sửa hyperparameter TCN rồi chỉ muốn chạy lại TCN-delta): dùng các cell riêng lẻ ở mục 7.1-7.4 bên dưới, không cần chạy lại cell gộp này.


In [54]:
import subprocess
import time

ALL_SCRIPTS = [
    # target V (Voltage), raw-target
    "main_lstm.py", "main_tcn.py", "main_seq2seq.py",
    # target V, Delta-Target
    "main_lstm_delta.py", "main_tcn_delta.py", "main_seq2seq_delta.py",
    # target W (Power), raw-target
    "main_lstm_power.py", "main_tcn_power.py", "main_seq2seq_power.py",
    # target W, Delta-Target
    "main_lstm_power_delta.py", "main_tcn_power_delta.py", "main_seq2seq_power_delta.py",
    # target V, Delta-Target + biet truoc I(t+h) (thi nghiem NARX)
    "main_lstm_delta_given_i.py", "main_tcn_delta_given_i.py", "main_seq2seq_delta_given_i.py",
]

SEP = "=" * 80
failed = []
t_start = time.time()
for i, script in enumerate(ALL_SCRIPTS, 1):
    print(SEP)
    print(f"[{i}/{len(ALL_SCRIPTS)}] {script}")
    print(SEP)
    t0 = time.time()
    result = subprocess.run(["python", script], cwd=SRC_DIR)
    elapsed = time.time() - t0
    if result.returncode != 0:
        print(f"!!! {script} THẤT BẠI (exit code {result.returncode}, {elapsed:.0f}s) -- tiếp tục script kế tiếp !!!")
        failed.append(script)
    else:
        print(f"--- {script} xong ({elapsed:.0f}s) ---")

print(SEP)
print(f"Tổng thời gian: {(time.time()-t_start)/60:.1f} phút")
if failed:
    print(f"CÁC SCRIPT LỖI ({len(failed)}): {failed}")
else:
    print("Tất cả 12 script chạy xong, không lỗi.")


[1/15] main_lstm.py
--- main_lstm.py xong (34s) ---
[2/15] main_tcn.py
--- main_tcn.py xong (64s) ---
[3/15] main_seq2seq.py
--- main_seq2seq.py xong (20s) ---
[4/15] main_lstm_delta.py
--- main_lstm_delta.py xong (33s) ---
[5/15] main_tcn_delta.py
--- main_tcn_delta.py xong (47s) ---
[6/15] main_seq2seq_delta.py
--- main_seq2seq_delta.py xong (18s) ---
[7/15] main_lstm_power.py
--- main_lstm_power.py xong (31s) ---
[8/15] main_tcn_power.py
--- main_tcn_power.py xong (59s) ---
[9/15] main_seq2seq_power.py
--- main_seq2seq_power.py xong (18s) ---
[10/15] main_lstm_power_delta.py
--- main_lstm_power_delta.py xong (26s) ---
[11/15] main_tcn_power_delta.py
--- main_tcn_power_delta.py xong (48s) ---
[12/15] main_seq2seq_power_delta.py
--- main_seq2seq_power_delta.py xong (19s) ---
[13/15] main_lstm_delta_given_i.py
--- main_lstm_delta_given_i.py xong (30s) ---
[14/15] main_tcn_delta_given_i.py
--- main_tcn_delta_given_i.py xong (50s) ---
[15/15] main_seq2seq_delta_given_i.py
--- main_seq2se

### 7.1 Chạy riêng — target Điện áp (V), raw-target

Mỗi model là 1 cell riêng, chạy tuần tự. Mỗi script tự: load + chuẩn bị data (`features.prepare_data()`), chia train/val/test theo run_id, tạo sliding window, train, đánh giá (MAE/RMSE/R2/DTW), rồi lưu model + predictions + bảng kết quả vào `outputs/` -- vì `outputs/` nằm ngay trong `DRIVE_PROJECT_DIR` (đang đứng trên Drive), mọi thứ tự động được giữ lại kể cả khi Colab ngắt kết nối (và không nằm trong git vì đã bị `.gitignore`).


In [55]:
%cd {SRC_DIR}
!python main_lstm.py


/content/drive/MyDrive/SOFC/src

=== LSTM h=1 ===
Epoch 1/150 - train_loss: 0.146268  val_loss: 0.002836
Epoch 2/150 - train_loss: 0.013525  val_loss: 0.003669
Epoch 3/150 - train_loss: 0.010364  val_loss: 0.002774
Epoch 4/150 - train_loss: 0.009092  val_loss: 0.002679
Epoch 5/150 - train_loss: 0.008854  val_loss: 0.002058
Epoch 6/150 - train_loss: 0.008005  val_loss: 0.004517
Epoch 7/150 - train_loss: 0.007780  val_loss: 0.001841
Epoch 8/150 - train_loss: 0.007690  val_loss: 0.002394
Epoch 9/150 - train_loss: 0.007263  val_loss: 0.005044
Epoch 10/150 - train_loss: 0.007656  val_loss: 0.001461
Epoch 11/150 - train_loss: 0.007421  val_loss: 0.002579
Epoch 12/150 - train_loss: 0.007191  val_loss: 0.003089
Epoch 13/150 - train_loss: 0.007079  val_loss: 0.001388
Epoch 14/150 - train_loss: 0.006946  val_loss: 0.002317
Epoch 15/150 - train_loss: 0.006904  val_loss: 0.002408
Epoch 16/150 - train_loss: 0.006651  val_loss: 0.001793
Epoch 17/150 - train_loss: 0.006482  val_loss: 0.001642
Epoch 1

In [56]:
%cd {SRC_DIR}
!python main_tcn.py


/content/drive/MyDrive/SOFC/src

=== TCN h=1 ===
Epoch 1/150 - train_loss: 0.199231  val_loss: 0.004547
Epoch 2/150 - train_loss: 0.018858  val_loss: 0.002514
Epoch 3/150 - train_loss: 0.012310  val_loss: 0.001753
Epoch 4/150 - train_loss: 0.011012  val_loss: 0.001756
Epoch 5/150 - train_loss: 0.009724  val_loss: 0.001639
Epoch 6/150 - train_loss: 0.009051  val_loss: 0.001593
Epoch 7/150 - train_loss: 0.008743  val_loss: 0.001469
Epoch 8/150 - train_loss: 0.008306  val_loss: 0.001471
Epoch 9/150 - train_loss: 0.008013  val_loss: 0.001747
Epoch 10/150 - train_loss: 0.007994  val_loss: 0.001333
Epoch 11/150 - train_loss: 0.007754  val_loss: 0.001213
Epoch 12/150 - train_loss: 0.007662  val_loss: 0.001502
Epoch 13/150 - train_loss: 0.007210  val_loss: 0.001353
Epoch 14/150 - train_loss: 0.007230  val_loss: 0.001144
Epoch 15/150 - train_loss: 0.006981  val_loss: 0.001222
Epoch 16/150 - train_loss: 0.007216  val_loss: 0.001202
Epoch 17/150 - train_loss: 0.006751  val_loss: 0.001108
Epoch 18

In [57]:
%cd {SRC_DIR}
!python main_seq2seq.py


/content/drive/MyDrive/SOFC/src
train window: (9081, 20, 33) (9081, 4)  test window: (2186, 20, 33) (2186, 4)
Epoch 1/150 - train_loss: 0.213294  val_loss: 0.007593
Epoch 2/150 - train_loss: 0.025058  val_loss: 0.003704
Epoch 3/150 - train_loss: 0.015383  val_loss: 0.003138
Epoch 4/150 - train_loss: 0.014307  val_loss: 0.002691
Epoch 5/150 - train_loss: 0.013843  val_loss: 0.003032
Epoch 6/150 - train_loss: 0.013129  val_loss: 0.003014
Epoch 7/150 - train_loss: 0.012609  val_loss: 0.003212
Epoch 8/150 - train_loss: 0.012209  val_loss: 0.005173
Epoch 9/150 - train_loss: 0.012062  val_loss: 0.004253
Epoch 10/150 - train_loss: 0.011814  val_loss: 0.004231
Epoch 11/150 - train_loss: 0.011749  val_loss: 0.005847
Epoch 12/150 - train_loss: 0.011485  val_loss: 0.005621
Epoch 13/150 - train_loss: 0.011334  val_loss: 0.005198
Epoch 14/150 - train_loss: 0.011267  val_loss: 0.005150
Early stopping at epoch 14 (no val improvement for 10 epochs)

--- horizon 1 ---
MAE  : 1.916814
RMSE : 2.623035
R^

### 7.2 Chạy riêng — target Điện áp (V), Delta-Target

Kết quả raw-target ở mục 7 cho thấy LSTM/TCN thắng đậm ở h=1 nhưng xuống dốc nhanh khi horizon tăng -- dấu hiệu persistence bias (model dựa nhiều vào `V_Lag1`, xem `notes/SOFC_data_notes.md` mục 14.3/16). Delta-Target Reformulation train model trên `y(t+h) - y(t)` thay vì giá trị thô, loại bỏ việc "chép Lag1" như một lối tắt miễn phí.

Baseline raw-target (`main_lstm.py`/`main_tcn.py`/`main_seq2seq.py`, mục 7) **không bị ghi đè** -- 3 script dưới đây ghi kết quả vào `*_delta_results.csv` riêng để so sánh trực tiếp 2 phiên bản.


In [58]:
%cd {SRC_DIR}
!python main_lstm_delta.py


/content/drive/MyDrive/SOFC/src

=== LSTM-delta h=1 ===
Epoch 1/150 - train_loss: 0.118013  val_loss: 0.027732
Epoch 2/150 - train_loss: 0.117249  val_loss: 0.027219
Epoch 3/150 - train_loss: 0.116759  val_loss: 0.027283
Epoch 4/150 - train_loss: 0.116277  val_loss: 0.027170
Epoch 5/150 - train_loss: 0.115955  val_loss: 0.027370
Epoch 6/150 - train_loss: 0.115655  val_loss: 0.027435
Epoch 7/150 - train_loss: 0.115438  val_loss: 0.027255
Epoch 8/150 - train_loss: 0.115166  val_loss: 0.027294
Epoch 9/150 - train_loss: 0.114692  val_loss: 0.027289
Epoch 10/150 - train_loss: 0.114521  val_loss: 0.027504
Epoch 11/150 - train_loss: 0.114493  val_loss: 0.027398
Epoch 12/150 - train_loss: 0.114148  val_loss: 0.027496
Epoch 13/150 - train_loss: 0.114000  val_loss: 0.027444
Epoch 14/150 - train_loss: 0.113877  val_loss: 0.027503
Early stopping at epoch 14 (no val improvement for 10 epochs)
MAE  : 0.197822
RMSE : 0.973024
R^2  : 0.992843
DTW  : 0.015316

=== LSTM-delta h=5 ===
Epoch 1/150 - train

In [59]:
%cd {SRC_DIR}
!python main_tcn_delta.py


/content/drive/MyDrive/SOFC/src

=== TCN-delta h=1 ===
Epoch 1/150 - train_loss: 0.117941  val_loss: 0.027312
Epoch 2/150 - train_loss: 0.117032  val_loss: 0.027126
Epoch 3/150 - train_loss: 0.116539  val_loss: 0.027144
Epoch 4/150 - train_loss: 0.115902  val_loss: 0.027371
Epoch 5/150 - train_loss: 0.115435  val_loss: 0.027248
Epoch 6/150 - train_loss: 0.115137  val_loss: 0.027417
Epoch 7/150 - train_loss: 0.114714  val_loss: 0.027262
Epoch 8/150 - train_loss: 0.114328  val_loss: 0.027285
Epoch 9/150 - train_loss: 0.114062  val_loss: 0.027794
Epoch 10/150 - train_loss: 0.113756  val_loss: 0.027475
Epoch 11/150 - train_loss: 0.113567  val_loss: 0.027687
Epoch 12/150 - train_loss: 0.113239  val_loss: 0.027591
Early stopping at epoch 12 (no val improvement for 10 epochs)
MAE  : 0.205077
RMSE : 0.971363
R^2  : 0.992868
DTW  : 0.015692

=== TCN-delta h=5 ===
Epoch 1/150 - train_loss: 0.183944  val_loss: 0.029916
Epoch 2/150 - train_loss: 0.180208  val_loss: 0.030198
Epoch 3/150 - train_los

In [60]:
%cd {SRC_DIR}
!python main_seq2seq_delta.py


/content/drive/MyDrive/SOFC/src
train window: (9081, 20, 33) (9081, 4)  test window: (2186, 20, 33) (2186, 4)
Epoch 1/150 - train_loss: 0.204607  val_loss: 0.030366
Epoch 2/150 - train_loss: 0.198953  val_loss: 0.032476
Epoch 3/150 - train_loss: 0.186534  val_loss: 0.027807
Epoch 4/150 - train_loss: 0.173885  val_loss: 0.032559
Epoch 5/150 - train_loss: 0.169552  val_loss: 0.028436
Epoch 6/150 - train_loss: 0.165062  val_loss: 0.030599
Epoch 7/150 - train_loss: 0.162498  val_loss: 0.032488
Epoch 8/150 - train_loss: 0.158522  val_loss: 0.033196
Epoch 9/150 - train_loss: 0.156959  val_loss: 0.030861
Epoch 10/150 - train_loss: 0.154179  val_loss: 0.035039
Epoch 11/150 - train_loss: 0.153798  val_loss: 0.033724
Epoch 12/150 - train_loss: 0.151287  val_loss: 0.035453
Epoch 13/150 - train_loss: 0.150040  val_loss: 0.034210
Early stopping at epoch 13 (no val improvement for 10 epochs)

--- horizon 1 ---
MAE  : 0.281103
RMSE : 0.973938
R^2  : 0.992807
DTW  : 0.046013

--- horizon 5 ---
MAE  : 

### 7.3 Chạy riêng — target Điện áp (V), Delta-Target + biết trước I(t+h) (thực nghiệm NARX)

Thực nghiệm 3 (`notes/SOFC_data_notes.md` mục 28-29): bản RF/XGBoost (local) cho thấy biết trước `I(t+h)` (dòng điện tương lai — thứ 1 bộ điều khiển thật sẽ TỰ BIẾT vì đó là setpoint do chính nó đặt ra) giúp rõ ở h=1 nhưng mất tác dụng từ h≥10, có thể vì cách nhét thẳng vào vector phẳng quá thô. 3 cell dưới đây thử lại trên LSTM/TCN/Seq2Seq — `I(t+h)` được thêm như 1 kênh feature riêng, broadcast dọc theo window (không làm phẳng), gần với cách paper NARX (Tofigh/Salehi 2024, mục 24) xử lý input ngoại sinh hơn.


In [61]:
%cd {SRC_DIR}
!python main_lstm_delta_given_i.py


/content/drive/MyDrive/SOFC/src

=== LSTM-delta-given-I h=1 ===
Epoch 1/150 - train_loss: 0.118094  val_loss: 0.027160
Epoch 2/150 - train_loss: 0.117109  val_loss: 0.027261
Epoch 3/150 - train_loss: 0.116572  val_loss: 0.027328
Epoch 4/150 - train_loss: 0.115820  val_loss: 0.027184
Epoch 5/150 - train_loss: 0.114922  val_loss: 0.027357
Epoch 6/150 - train_loss: 0.114135  val_loss: 0.027540
Epoch 7/150 - train_loss: 0.113589  val_loss: 0.027596
Epoch 8/150 - train_loss: 0.113113  val_loss: 0.027603
Epoch 9/150 - train_loss: 0.112706  val_loss: 0.027486
Epoch 10/150 - train_loss: 0.112066  val_loss: 0.027949
Epoch 11/150 - train_loss: 0.111774  val_loss: 0.028579
Early stopping at epoch 11 (no val improvement for 10 epochs)
MAE  : 0.206269
RMSE : 0.974039
R^2  : 0.992828
DTW  : 0.015377

=== LSTM-delta-given-I h=5 ===
Epoch 1/150 - train_loss: 0.183821  val_loss: 0.029282
Epoch 2/150 - train_loss: 0.178967  val_loss: 0.031626
Epoch 3/150 - train_loss: 0.174186  val_loss: 0.029040
Epoch 

In [62]:
%cd {SRC_DIR}
!python main_tcn_delta_given_i.py


/content/drive/MyDrive/SOFC/src

=== TCN-delta-given-I h=1 ===
Epoch 1/150 - train_loss: 0.120212  val_loss: 0.027149
Epoch 2/150 - train_loss: 0.116974  val_loss: 0.027324
Epoch 3/150 - train_loss: 0.116199  val_loss: 0.027143
Epoch 4/150 - train_loss: 0.114653  val_loss: 0.027394
Epoch 5/150 - train_loss: 0.113555  val_loss: 0.027576
Epoch 6/150 - train_loss: 0.111553  val_loss: 0.028133
Epoch 7/150 - train_loss: 0.108764  val_loss: 0.027436
Epoch 8/150 - train_loss: 0.105254  val_loss: 0.028327
Epoch 9/150 - train_loss: 0.103418  val_loss: 0.028468
Epoch 10/150 - train_loss: 0.100661  val_loss: 0.028632
Epoch 11/150 - train_loss: 0.097826  val_loss: 0.029002
Epoch 12/150 - train_loss: 0.096156  val_loss: 0.027676
Epoch 13/150 - train_loss: 0.094808  val_loss: 0.028072
Early stopping at epoch 13 (no val improvement for 10 epochs)
MAE  : 0.200827
RMSE : 0.952214
R^2  : 0.993146
DTW  : 0.022070

=== TCN-delta-given-I h=5 ===
Epoch 1/150 - train_loss: 0.185823  val_loss: 0.028822
Epoch 

In [63]:
%cd {SRC_DIR}
!python main_seq2seq_delta_given_i.py


/content/drive/MyDrive/SOFC/src
train window: (9081, 20, 37) (9081, 4)  test window: (2186, 20, 37) (2186, 4)
Epoch 1/150 - train_loss: 0.203637  val_loss: 0.030745
Epoch 2/150 - train_loss: 0.198445  val_loss: 0.029309
Epoch 3/150 - train_loss: 0.186625  val_loss: 0.029436
Epoch 4/150 - train_loss: 0.172740  val_loss: 0.030149
Epoch 5/150 - train_loss: 0.166054  val_loss: 0.029025
Epoch 6/150 - train_loss: 0.164633  val_loss: 0.030592
Epoch 7/150 - train_loss: 0.160695  val_loss: 0.032006
Epoch 8/150 - train_loss: 0.156783  val_loss: 0.034009
Epoch 9/150 - train_loss: 0.154523  val_loss: 0.039408
Epoch 10/150 - train_loss: 0.150339  val_loss: 0.039242
Epoch 11/150 - train_loss: 0.148229  val_loss: 0.047399
Epoch 12/150 - train_loss: 0.146723  val_loss: 0.038821
Epoch 13/150 - train_loss: 0.144701  val_loss: 0.036676
Epoch 14/150 - train_loss: 0.143619  val_loss: 0.040774
Epoch 15/150 - train_loss: 0.141785  val_loss: 0.039155
Early stopping at epoch 15 (no val improvement for 10 epoch

### 7.4 Chạy riêng — target Công suất (W), raw-target

Mở rộng sang dự đoán `W` (Power) thay vì `V` (xem `notes/SOFC_data_notes.md` mục 25) — cùng pipeline, đổi target qua `features.prepare_data(target="W")`. `V` và `I` được giữ lại làm feature (không phải leakage, khác với lý do loại `W` khi dự đoán `V`). Kết quả RF/XGBoost-Power (local) cho thấy đây là bài toán khó hơn `V` rõ rệt (R² thấp hơn, giảm nhanh hơn theo horizon) — 3 cell dưới đây kiểm tra xem LSTM/TCN/Seq2Seq có làm tốt hơn không.


In [64]:
%cd {SRC_DIR}
!python main_lstm_power.py


/content/drive/MyDrive/SOFC/src

=== LSTM-power h=1 ===
Epoch 1/150 - train_loss: 0.127439  val_loss: 0.000401
Epoch 2/150 - train_loss: 0.075447  val_loss: 0.000051
Epoch 3/150 - train_loss: 0.037827  val_loss: 0.000093
Epoch 4/150 - train_loss: 0.035779  val_loss: 0.000009
Epoch 5/150 - train_loss: 0.034107  val_loss: 0.000006
Epoch 6/150 - train_loss: 0.031718  val_loss: 0.000021
Epoch 7/150 - train_loss: 0.033082  val_loss: 0.000048
Epoch 8/150 - train_loss: 0.031753  val_loss: 0.000033
Epoch 9/150 - train_loss: 0.031832  val_loss: 0.000005
Epoch 10/150 - train_loss: 0.030333  val_loss: 0.000021
Epoch 11/150 - train_loss: 0.028847  val_loss: 0.000019
Epoch 12/150 - train_loss: 0.028176  val_loss: 0.000032
Epoch 13/150 - train_loss: 0.028095  val_loss: 0.000033
Epoch 14/150 - train_loss: 0.025207  val_loss: 0.000005
Epoch 15/150 - train_loss: 0.025470  val_loss: 0.000011
Epoch 16/150 - train_loss: 0.024981  val_loss: 0.000020
Epoch 17/150 - train_loss: 0.025601  val_loss: 0.000025
E

In [65]:
%cd {SRC_DIR}
!python main_tcn_power.py


/content/drive/MyDrive/SOFC/src

=== TCN-power h=1 ===
Epoch 1/150 - train_loss: 0.134977  val_loss: 0.000281
Epoch 2/150 - train_loss: 0.073760  val_loss: 0.000694
Epoch 3/150 - train_loss: 0.038151  val_loss: 0.000166
Epoch 4/150 - train_loss: 0.036588  val_loss: 0.000162
Epoch 5/150 - train_loss: 0.034388  val_loss: 0.000090
Epoch 6/150 - train_loss: 0.033838  val_loss: 0.000067
Epoch 7/150 - train_loss: 0.031852  val_loss: 0.000220
Epoch 8/150 - train_loss: 0.031368  val_loss: 0.000169
Epoch 9/150 - train_loss: 0.031048  val_loss: 0.000093
Epoch 10/150 - train_loss: 0.030276  val_loss: 0.000245
Epoch 11/150 - train_loss: 0.029344  val_loss: 0.000072
Epoch 12/150 - train_loss: 0.028069  val_loss: 0.000074
Epoch 13/150 - train_loss: 0.028649  val_loss: 0.000048
Epoch 14/150 - train_loss: 0.027346  val_loss: 0.000038
Epoch 15/150 - train_loss: 0.027420  val_loss: 0.000042
Epoch 16/150 - train_loss: 0.027900  val_loss: 0.000186
Epoch 17/150 - train_loss: 0.026229  val_loss: 0.000049
Ep

In [66]:
%cd {SRC_DIR}
!python main_seq2seq_power.py


/content/drive/MyDrive/SOFC/src
train window: (9081, 20, 34) (9081, 4)  test window: (2186, 20, 34) (2186, 4)
Epoch 1/150 - train_loss: 0.140019  val_loss: 0.000086
Epoch 2/150 - train_loss: 0.116914  val_loss: 0.000037
Epoch 3/150 - train_loss: 0.069702  val_loss: 0.000046
Epoch 4/150 - train_loss: 0.057542  val_loss: 0.000130
Epoch 5/150 - train_loss: 0.056280  val_loss: 0.000196
Epoch 6/150 - train_loss: 0.053979  val_loss: 0.000246
Epoch 7/150 - train_loss: 0.051156  val_loss: 0.000250
Epoch 8/150 - train_loss: 0.050280  val_loss: 0.000163
Epoch 9/150 - train_loss: 0.049346  val_loss: 0.000240
Epoch 10/150 - train_loss: 0.048134  val_loss: 0.000235
Epoch 11/150 - train_loss: 0.047764  val_loss: 0.000204
Epoch 12/150 - train_loss: 0.047375  val_loss: 0.000200
Early stopping at epoch 12 (no val improvement for 10 epochs)

--- horizon 1 ---
MAE  : 36.139026
RMSE : 147.494641
R^2  : 0.505261
DTW  : 14.399716

--- horizon 5 ---
MAE  : 39.962175
RMSE : 163.114904
R^2  : 0.394923
DTW  : 1

### 7.5 Chạy riêng — target Công suất (W), Delta-Target

RF/XGBoost-Power-delta (local, mục 25.5) vẫn thua persistence baseline ở mọi horizon — chưa xác nhận được giả thuyết "Power là nơi Delta-Target có giá trị thực tiễn nhất" (persistence của Power yếu hẳn ở horizon dài, R²≈0.008 ở h=20 — nhiều dư địa hơn cho model thật). Đây là phép thử quyết định: LSTM/TCN/Seq2Seq-delta từng thắng dần persistence theo horizon trên `V` (mục 22.3) — xem có lặp lại trên `W` không.


In [67]:
%cd {SRC_DIR}
!python main_lstm_power_delta.py


/content/drive/MyDrive/SOFC/src

=== LSTM-power-delta h=1 ===
Epoch 1/150 - train_loss: 0.072667  val_loss: 0.000006
Epoch 2/150 - train_loss: 0.072573  val_loss: 0.000007
Epoch 3/150 - train_loss: 0.072554  val_loss: 0.000001
Epoch 4/150 - train_loss: 0.072549  val_loss: 0.000005
Epoch 5/150 - train_loss: 0.072493  val_loss: 0.000001
Epoch 6/150 - train_loss: 0.072488  val_loss: 0.000006
Epoch 7/150 - train_loss: 0.072478  val_loss: 0.000001
Epoch 8/150 - train_loss: 0.072432  val_loss: 0.000004
Epoch 9/150 - train_loss: 0.072442  val_loss: 0.000001
Epoch 10/150 - train_loss: 0.072383  val_loss: 0.000001
Epoch 11/150 - train_loss: 0.072355  val_loss: 0.000002
Epoch 12/150 - train_loss: 0.072305  val_loss: 0.000001
Epoch 13/150 - train_loss: 0.072259  val_loss: 0.000010
Epoch 14/150 - train_loss: 0.072270  val_loss: 0.000001
Epoch 15/150 - train_loss: 0.072240  val_loss: 0.000001
Epoch 16/150 - train_loss: 0.072219  val_loss: 0.000011
Epoch 17/150 - train_loss: 0.072197  val_loss: 0.00

In [68]:
%cd {SRC_DIR}
!python main_tcn_power_delta.py


/content/drive/MyDrive/SOFC/src

=== TCN-power-delta h=1 ===
Epoch 1/150 - train_loss: 0.074409  val_loss: 0.000029
Epoch 2/150 - train_loss: 0.072638  val_loss: 0.000034
Epoch 3/150 - train_loss: 0.072439  val_loss: 0.000022
Epoch 4/150 - train_loss: 0.072451  val_loss: 0.000008
Epoch 5/150 - train_loss: 0.072327  val_loss: 0.000003
Epoch 6/150 - train_loss: 0.072213  val_loss: 0.000017
Epoch 7/150 - train_loss: 0.072243  val_loss: 0.000030
Epoch 8/150 - train_loss: 0.072104  val_loss: 0.000012
Epoch 9/150 - train_loss: 0.072046  val_loss: 0.000072
Epoch 10/150 - train_loss: 0.071922  val_loss: 0.000028
Epoch 11/150 - train_loss: 0.071840  val_loss: 0.000009
Epoch 12/150 - train_loss: 0.071858  val_loss: 0.000055
Epoch 13/150 - train_loss: 0.071801  val_loss: 0.000043
Epoch 14/150 - train_loss: 0.071771  val_loss: 0.000024
Epoch 15/150 - train_loss: 0.071635  val_loss: 0.000014
Early stopping at epoch 15 (no val improvement for 10 epochs)
MAE  : 7.220698
RMSE : 61.968254
R^2  : 0.9119

In [69]:
%cd {SRC_DIR}
!python main_seq2seq_power_delta.py


/content/drive/MyDrive/SOFC/src
train window: (9081, 20, 34) (9081, 4)  test window: (2186, 20, 34) (2186, 4)
Epoch 1/150 - train_loss: 0.106341  val_loss: 0.000004
Epoch 2/150 - train_loss: 0.105268  val_loss: 0.000005
Epoch 3/150 - train_loss: 0.105211  val_loss: 0.000001
Epoch 4/150 - train_loss: 0.105094  val_loss: 0.000001
Epoch 5/150 - train_loss: 0.104983  val_loss: 0.000059
Epoch 6/150 - train_loss: 0.104789  val_loss: 0.000047
Epoch 7/150 - train_loss: 0.104509  val_loss: 0.000066
Epoch 8/150 - train_loss: 0.104012  val_loss: 0.000020
Epoch 9/150 - train_loss: 0.103534  val_loss: 0.000039
Epoch 10/150 - train_loss: 0.103186  val_loss: 0.000128
Epoch 11/150 - train_loss: 0.102851  val_loss: 0.000084
Epoch 12/150 - train_loss: 0.102519  val_loss: 0.000381
Epoch 13/150 - train_loss: 0.102063  val_loss: 0.000179
Epoch 14/150 - train_loss: 0.101963  val_loss: 0.000141
Early stopping at epoch 14 (no val improvement for 10 epochs)

--- horizon 1 ---
MAE  : 7.061314
RMSE : 62.274075
R

## 7.6 Thực nghiệm 3, phiên bản nhiều seed (trả lời dứt điểm câu hỏi mục 7.3)

Mục 7.3 chỉ chạy 1 lần mỗi cấu hình (có/không biết trước `I(t+h)`) — không đủ để phân biệt hiệu ứng thật với nhiễu tự nhiên giữa các lần train LSTM/TCN/Seq2Seq (`notes/SOFC_data_notes.md` mục 31). Cell dưới đây lặp lại **5 seed** (42-46) cho mỗi cấu hình, tính trung bình ± độ lệch chuẩn, rồi tự động kết luận "thật" hay "nhiễu" bằng cách so khoảng cách 2 trung bình với tổng 2 độ lệch chuẩn.

Ước tính ~15-25 phút (3 kiến trúc × 2 biến thể × 5 seed × 4 horizon, LSTM/TCN train riêng từng horizon còn Seq2Seq train 1 lần cho cả 4). Kết quả lưu 3 file: `given_i_multiseed_raw.csv` (từng lần chạy), `given_i_multiseed_summary.csv` (mean±std), `given_i_multiseed_verdict.csv` (kết luận từng horizon/kiến trúc).


In [70]:
%cd {SRC_DIR}
!python main_given_i_multiseed.py


/content/drive/MyDrive/SOFC/src
[1/30] LSTM/baseline/seed=42 (31s): h1=0.198, h5=0.714, h10=1.073, h20=1.548
[2/30] LSTM/baseline/seed=43 (28s): h1=0.193, h5=0.762, h10=1.141, h20=1.617
[3/30] LSTM/baseline/seed=44 (26s): h1=0.213, h5=0.707, h10=1.084, h20=1.526
[4/30] LSTM/baseline/seed=45 (28s): h1=0.196, h5=0.705, h10=1.088, h20=1.540
[5/30] LSTM/baseline/seed=46 (26s): h1=0.194, h5=0.750, h10=1.079, h20=1.534
[6/30] LSTM/given_I/seed=42 (26s): h1=0.206, h5=0.628, h10=1.109, h20=1.535
[7/30] LSTM/given_I/seed=43 (23s): h1=0.198, h5=0.652, h10=1.082, h20=1.464
[8/30] LSTM/given_I/seed=44 (30s): h1=0.197, h5=0.555, h10=0.975, h20=1.596
[9/30] LSTM/given_I/seed=45 (24s): h1=0.200, h5=0.672, h10=0.979, h20=1.528
[10/30] LSTM/given_I/seed=46 (22s): h1=0.205, h5=0.664, h10=1.154, h20=1.563
[11/30] TCN/baseline/seed=42 (43s): h1=0.205, h5=0.657, h10=1.176, h20=1.567
[12/30] TCN/baseline/seed=43 (47s): h1=0.201, h5=0.677, h10=1.135, h20=1.676
[13/30] TCN/baseline/seed=44 (44s): h1=0.205, h5

## 7.7 Multi-seed cho kết luận CHÍNH (Delta-target V vs persistence)

Kết luận cốt lõi của cả project (mục 19/22, `notes/SOFC_data_notes.md`) — Seq2Seq-delta thắng persistence, biên độ tăng dần theo horizon — trước giờ chỉ dựa trên **1 lần chạy duy nhất**. Cell dưới đây lặp lại LSTM/TCN/Seq2Seq-delta qua 5 seed (persistence là số cố định, không cần seed), so mean±std của model với persistence, tự kết luận thắng/thua có vượt nhiễu train hay không — cùng logic đã dùng ở mục 7.6.

Ước tính ~10-15 phút. Kết quả: `delta_vs_persistence_multiseed_{raw,summary,verdict}.csv`.


In [71]:
%cd {SRC_DIR}
!python main_delta_vs_persistence_multiseed.py


/content/drive/MyDrive/SOFC/src
Persistence MAE (fixed, no seed): {1: 0.2014, 5: 0.7025, 10: 1.1434, 20: 1.6899}
[1/15] LSTM-delta/seed=42 (31s): h1=0.198, h5=0.714, h10=1.073, h20=1.548
[2/15] LSTM-delta/seed=43 (28s): h1=0.193, h5=0.762, h10=1.141, h20=1.617
[3/15] LSTM-delta/seed=44 (24s): h1=0.213, h5=0.707, h10=1.084, h20=1.526
[4/15] LSTM-delta/seed=45 (28s): h1=0.196, h5=0.705, h10=1.088, h20=1.540
[5/15] LSTM-delta/seed=46 (25s): h1=0.194, h5=0.750, h10=1.079, h20=1.534
[6/15] TCN-delta/seed=42 (41s): h1=0.205, h5=0.657, h10=1.176, h20=1.567
[7/15] TCN-delta/seed=43 (46s): h1=0.201, h5=0.677, h10=1.135, h20=1.676
[8/15] TCN-delta/seed=44 (42s): h1=0.205, h5=0.682, h10=1.114, h20=1.581
[9/15] TCN-delta/seed=45 (42s): h1=0.203, h5=0.658, h10=1.070, h20=1.600
[10/15] TCN-delta/seed=46 (48s): h1=0.202, h5=0.701, h10=1.058, h20=1.563
[11/15] Seq2Seq-delta/seed=42 (15s): h1=0.281, h5=0.728, h10=1.125, h20=1.488
[12/15] Seq2Seq-delta/seed=43 (23s): h1=0.271, h5=0.777, h10=1.327, h20=1

## 8. So sánh tất cả model (Colab) — V và W, raw và delta

Đọc lại `outputs/reports/*.csv` để so sánh, không cần chạy lại cell nào ở mục 7.


In [72]:
import pandas as pd

REPORTS_DIR = f"{DRIVE_PROJECT_DIR}/outputs/reports"

combined = []
for name, fname in [
    ("LSTM", "lstm_results.csv"), ("TCN", "tcn_results.csv"), ("Seq2Seq", "seq2seq_results.csv"),
    ("LSTM-delta", "lstm_delta_results.csv"), ("TCN-delta", "tcn_delta_results.csv"), ("Seq2Seq-delta", "seq2seq_delta_results.csv"),
    ("LSTM-power", "lstm_power_results.csv"), ("TCN-power", "tcn_power_results.csv"), ("Seq2Seq-power", "seq2seq_power_results.csv"),
    ("LSTM-power-delta", "lstm_power_delta_results.csv"), ("TCN-power-delta", "tcn_power_delta_results.csv"), ("Seq2Seq-power-delta", "seq2seq_power_delta_results.csv"),
    ("LSTM-delta-given-I", "lstm_delta_given_i_results.csv"), ("TCN-delta-given-I", "tcn_delta_given_i_results.csv"), ("Seq2Seq-delta-given-I", "seq2seq_delta_given_i_results.csv"),
]:
    path = f"{REPORTS_DIR}/{fname}"
    if os.path.exists(path):
        d = pd.read_csv(path)
        d.insert(0, "model", name)
        combined.append(d)
    else:
        print(f"(chưa có {fname} -- chạy cell tương ứng ở mục 7 trước)")

combined_df = pd.concat(combined, ignore_index=True) if combined else pd.DataFrame()
combined_df


,model,horizon,MAE,RMSE,R2,DTW
0,LSTM,1,1.120616,1.782173,0.975991,0.419769
1,LSTM,5,1.724253,2.632270,0.946197,0.668122
2,LSTM,10,2.635778,3.751577,0.886829,1.125873
3,LSTM,20,2.680606,3.743897,0.878571,1.142941
4,TCN,1,1.322699,2.033598,0.968739,0.528082
5,TCN,5,2.175174,3.039533,0.928260,0.905852
6,TCN,10,2.307887,3.393101,0.907424,0.952251
7,TCN,20,2.701824,3.840977,0.872192,1.146055
8,Seq2Seq,1,1.916814,2.623035,0.947826,0.807001
9,Seq2Seq,5,1.986866,2.865836,0.936037,0.774254


## 9. Ghi chú

- Mỗi lần **sửa code ở máy local**, nhớ `git push` trước khi mở lại Colab, rồi chạy lại Cell 3 (`!git pull`) để đồng bộ -- không cần upload tay nữa.
- Kết quả (`outputs/reports/*.csv`), model (`outputs/models_saved/`), và predictions (`outputs/predictions_cache/`) đều nằm trên Drive, **không nằm trong git** (`.gitignore`) -> tải thủ công về `E:\sofc\outputs\` nếu muốn gộp chung với kết quả Random Forest / XGBoost chạy local (`src/main.py`).
- Random Forest và XGBoost KHÔNG chạy ở đây (không cần GPU, chạy local nhanh hơn nhiều qua `python src/main.py`).
- Hyperparameter (`hidden_size`, `num_channels`, `epochs`, `patience`...) trong `main_lstm.py`/`main_tcn.py`/`main_seq2seq.py` là điểm khởi đầu, chưa tune riêng theo dataset SOFC -- nếu train loss không giảm hoặc R2 âm, sửa trực tiếp trong script đó ở local, `git push`, rồi `git pull` lại trên Colab và chạy lại cell tương ứng.
- Muốn train lại 1 model cụ thể: chỉ cần chạy lại đúng cell của model đó ở mục 7/7b, không cần chạy lại toàn bộ notebook.

- Kết quả target Power (`*_power_results.csv`, `*_power_delta_results.csv`) tải về `E:\sofc\outputs\` giống hệt cách làm với target Voltage (mục 20 trong notes).